导出测试向量图片

In [ ]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 0. 定义手动展平函数
# ==========================================

def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    假设输入 x 的形状为 (Batch, Channels, Height, Width)
    """
    B, C, H, W = x.shape
    
    # 准备一个普通的 Python 列表，充当我们的“一维连续内存”
    flattened_memory = []
    
    # 严格按照行优先的多重循环读取数据
    # 最外层是 Batch，然后是 Channel，接着是 Height，最内层是连续的 Width
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    # 每次读取一个具体的数值，追加到一维内存中
                    # 这个遍历顺序完美等价于我们之前推导的 C 语言一维索引计算公式：
                    # Index = c * (H * W) + h * W + w
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
                    
    # 将我们手工铺平的一维列表重新包装成 PyTorch 张量
    # 形状为 (B, C * H * W)，在我们的例子中就是 (1, 2048)
    # x.dtype 和 x.device 确保数据类型和所在硬件（CPU/GPU）与原输入保持一致
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

# 1. 再次定义那个用来加载权重的空壳网络
class HardwareFriendlyCNN(torch.nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = torch.nn.Linear(32 * 8 * 8, 10)
        self.relu = torch.nn.ReLU()
    def forward(self, x):
        pass

def sym_quantize(tensor):
    """对称量化：强制 Zero Point = 0"""
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -128, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    """导出为一维 C 数组"""
    flat_data = tensor.flatten().tolist()
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ")
    for i, val in enumerate(flat_data):
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def main():
    # 1. 加载模型
    model = HardwareFriendlyCNN()
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_69p.pth', weights_only=True))
    model.eval()

    # 2. 从 CIFAR-10 测试集提取第 0 张图片（一般是一只猫）
    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)
    img_fp32, label = testset[0]
    img_fp32 = img_fp32.unsqueeze(0) # [1, 3, 32, 32]

    print(f"提取测试图片成功！真实标签: {label}")

    # 3. 对输入、权重进行对称量化 (ZP=0)
    w_fp32 = model.conv1.weight.data
    b_fp32 = model.conv1.bias.data

    img_q, s_in = sym_quantize(img_fp32)
    w_q, s_w = sym_quantize(w_fp32)

    # 偏置量化：Scale 必须是 s_in * s_w，且存为 INT32
    s_bias = s_in * s_w
    b_q = torch.round(b_fp32 / s_bias).to(torch.int32)

    # 4. 计算 FP32 理论输出并量化，得到输出的 Scale
    out_fp32 = F.conv2d(img_fp32, w_fp32, b_fp32, padding=1)
    _, s_out = sym_quantize(out_fp32)
    out_zp = 0 # 依然保持对称量化

    # 5. 计算连接 C 语言硬件逻辑的乘数(Multiplier)和移位数(Shift)
    M = (s_in * s_w) / s_out
    shift = 16  # 固定右移 16 位，硬件极其容易实现
    multiplier = int(round(M * (1 << shift)))

    # 6. 【核心】在 Python 中完全模拟 C 语言的整数乘加逻辑，生成绝对标准的答案！
    # 因为浮点计算没有溢出，我们直接把量化后的 INT8 转成 Float 算，结果等同于精确的 INT32 算术
    exact_mac_int32 = F.conv2d(img_q.float(), w_q.float(), bias=b_q.float(), padding=1).to(torch.int32)
    
    # 严格遵循 C 语言的 Re-quantization 逻辑
    scaled_acc = (exact_mac_int32 * multiplier) >> shift
    expected_out_q = torch.clamp(scaled_acc + out_zp, -128, 127).to(torch.int8)

    # # 7. 导出到 test_vector.h
    # print("正在生成 test_vector.h...")
    # with open("test_vector.h", "w") as f:
    #     f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n")
    #     f.write("#include <stdint.h>\n\n")
    #     f.write(f"// 量化参数: Multiplier = {multiplier}, Shift = {shift}, Out_ZP = {out_zp}\n\n")
        
    #     export_c_array(img_q, "test_input_img", f, "int8_t")
    #     export_c_array(w_q, "test_conv1_weight", f, "int8_t")
    #     export_c_array(b_q, "test_conv1_bias", f, "int32_t")
    #     export_c_array(expected_out_q, "expected_conv1_output", f, "int8_t")
        
    #     f.write("#endif // TEST_VECTOR_H\n")

    print(f"完成！请注意：Multiplier={multiplier}, Shift={shift}")

if __name__ == "__main__":
    main()

提取测试图片成功！真实标签: 3
完成！请注意：Multiplier=151, Shift=16
